In [9]:
import requests

def fetch_uniprot(query, total=1000):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": query, "format": "fasta", "size": 500}
    all_fasta = []
    url = base_url

    while url and ''.join(all_fasta).count('>') < total:
        response = requests.get(url, params=params)
        all_fasta.append(response.text)
        url = response.links.get("next", {}).get("url")
        params = {}
        print(f"  Fetched {''.join(all_fasta).count('>')} sequences so far...")

    return ''.join(all_fasta)

signal_fasta = fetch_uniprot("ft_signal:* AND reviewed:true", total=1000)
nonsignal_fasta = fetch_uniprot("(cc_scl_term:Cytoplasm) AND (reviewed:true) AND (NOT ft_signal:*)", total=1000)

with open("Data/signal_peptides.fasta", "w") as f:
    f.write(signal_fasta)
with open("Data/nonsignal_peptides.fasta", "w") as f:
    f.write(nonsignal_fasta)

print(f"Signal: {signal_fasta.count('>')}")
print(f"Non-signal: {nonsignal_fasta.count('>')}")

  Fetched 500 sequences so far...
  Fetched 1000 sequences so far...
  Fetched 500 sequences so far...
  Fetched 1000 sequences so far...
Signal: 1000
Non-signal: 1000


In [7]:
# check for duplicate headers
from Bio import SeqIO

def check_duplicates(filepath):
    ids = [record.id for record in SeqIO.parse(filepath, "fasta")]
    unique = set(ids)
    print(f"Total sequences : {len(ids)}")
    print(f"Unique sequences: {len(unique)}")
    print(f"Duplicates      : {len(ids) - len(unique)}")

check_duplicates("Data/signal_peptides.fasta")
check_duplicates("Data/nonsignal_peptides.fasta")

Total sequences : 1000
Unique sequences: 1000
Duplicates      : 0
Total sequences : 1000
Unique sequences: 1000
Duplicates      : 0
